# CICIDS2017 Colab ML Train + Eval

This notebook trains CICIDS2017 machine-learning baseline artifacts on Google Colab, evaluates the trained models on the held-out test split, writes the final ML metrics JSON, verifies artifacts, and exports a Drive backup zip.

Expected final metric output: `results/cicids2017_multi_ml_baselines.json`. This file is what the final evaluation notebook uses for CICIDS2017 ML rows.

In [8]:
from pathlib import Path
import os
import sys
import subprocess

REPO_URL = "https://github.com/HoangZuzi-14/Nids_deep_learning.git"
CLONE_DIR = Path("/content/Nids_deep_learning")

def find_project_dir():
    candidates = [
        Path.cwd(),
        Path.cwd() / "ids_deep_learning",
        CLONE_DIR,
        CLONE_DIR / "ids_deep_learning",
        Path("/content/ids_deep_learning"),
        Path("/content/ids_deep_learning") / "ids_deep_learning",
    ]
    for candidate in candidates:
        if (candidate / "src").exists() and (candidate / "config").exists():
            return candidate
    return None

PROJECT_DIR = find_project_dir()
if PROJECT_DIR is None:
    subprocess.run(["git", "clone", REPO_URL, str(CLONE_DIR)], check=True)
    PROJECT_DIR = find_project_dir()

if PROJECT_DIR is None:
    raise RuntimeError("Could not locate project directory containing src/ and config/.")

for path in [CLONE_DIR, PROJECT_DIR, PROJECT_DIR.parent]:
    if path.exists() and str(path) not in sys.path:
        sys.path.insert(0, str(path))

os.chdir(PROJECT_DIR)
print(f"Project directory: {PROJECT_DIR}")
print("Python search path head:", sys.path[:4])

Project directory: /content/Nids_deep_learning/ids_deep_learning
Python search path head: ['/content/Nids_deep_learning/ids_deep_learning', '/content/Nids_deep_learning', '/content', '/env/python']


In [9]:
%pip install -q -r requirements.txt

In [10]:
from google.colab import drive
# This will prompt for authorization to access your Google Drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


## Raw Data Placement

Recommended: keep the eight CICIDS2017 raw CSV files in Google Drive at `/content/drive/MyDrive/nids_data/cicids2017/raw` or inside the project raw folder. The notebook also accepts `/content/cicids2017_raw`, but files in `/content` disappear when the Colab runtime resets.

If you have a zip containing the eight CSV files, put the zip in `/content` or `/content/drive/MyDrive/nids_data/`; the notebook will try to extract it automatically.

In [11]:
from pathlib import Path

EXPECTED_FILES = [
    "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv",
    "Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv",
    "Friday-WorkingHours-Morning.pcap_ISCX.csv",
    "Monday-WorkingHours.pcap_ISCX.csv",
    "Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv",
    "Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv",
    "Tuesday-WorkingHours.pcap_ISCX.csv",
    "Wednesday-workingHours.pcap_ISCX.csv",
]

CANDIDATE_RAW_DIRS = [
    Path("/content/drive/MyDrive/cicids2017"), # Added user specified path
    Path("/content/drive/MyDrive/nids_deep_learning/ids_deep_learning/data/raw/cicids2017"),
    Path("/content/drive/MyDrive/nids_data/cicids2017/raw"),
    Path("/content/drive/MyDrive/cicids2017_raw"),
    Path("/content/cicids2017_raw"),
    PROJECT_DIR / "data" / "raw" / "cicids2017",
]

CICIDS_RAW_DIR = None
CICIDS_CACHE_PATH = Path("/content/cicids2017_cache/merged.csv")
OUTPUT_ZIP = Path("/content/drive/MyDrive/nids_outputs/cicids2017_ml_artifacts.zip")

CLASSIFICATION = "multi"
MODELS_TO_TRAIN = ["LogisticRegression", "RandomForest", "XGBoost", "LightGBM"]

def missing_files(raw_dir: Path):
    return [name for name in EXPECTED_FILES if not (raw_dir / name).exists()]

def complete_raw_dir(raw_dir: Path):
    return raw_dir.exists() and not missing_files(raw_dir)

for candidate in CANDIDATE_RAW_DIRS:
    if complete_raw_dir(candidate):
        CICIDS_RAW_DIR = candidate
        break

Path("/content/cicids2017_raw").mkdir(parents=True, exist_ok=True)
print("Candidate raw directories:")
for candidate in CANDIDATE_RAW_DIRS:
    existing = sorted(p.name for p in candidate.glob("*.csv")) if candidate.exists() else []
    print(f"- {candidate} | exists={candidate.exists()} | csv_count={len(existing)}")

print("\nSelected raw dir:", CICIDS_RAW_DIR)
print("Cache path:", CICIDS_CACHE_PATH)
print("Output zip:", OUTPUT_ZIP)
print("Models:", MODELS_TO_TRAIN)

Candidate raw directories:
- /content/drive/MyDrive/cicids2017 | exists=True | csv_count=8
- /content/drive/MyDrive/nids_deep_learning/ids_deep_learning/data/raw/cicids2017 | exists=False | csv_count=0
- /content/drive/MyDrive/nids_data/cicids2017/raw | exists=False | csv_count=0
- /content/drive/MyDrive/cicids2017_raw | exists=False | csv_count=0
- /content/cicids2017_raw | exists=True | csv_count=0
- /content/Nids_deep_learning/ids_deep_learning/data/raw/cicids2017 | exists=False | csv_count=0

Selected raw dir: /content/drive/MyDrive/cicids2017
Cache path: /content/cicids2017_cache/merged.csv
Output zip: /content/drive/MyDrive/nids_outputs/cicids2017_ml_artifacts.zip
Models: ['LogisticRegression', 'RandomForest', 'XGBoost', 'LightGBM']


In [12]:
import zipfile

def zip_contains_expected(zip_path: Path):
    try:
        with zipfile.ZipFile(zip_path) as archive:
            names = {Path(info.filename).name for info in archive.infolist() if not info.is_dir()}
        return set(EXPECTED_FILES).issubset(names)
    except zipfile.BadZipFile:
        return False

def extract_expected_files(zip_path: Path, target_dir: Path):
    target_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as archive:
        members_by_name = {Path(info.filename).name: info for info in archive.infolist() if not info.is_dir()}
        for expected in EXPECTED_FILES:
            info = members_by_name[expected]
            with archive.open(info) as src, (target_dir / expected).open("wb") as dst:
                dst.write(src.read())

# Only attempt to extract from zips if CICIDS_RAW_DIR is not already set
if CICIDS_RAW_DIR is None:
    zip_candidates = []
    for base in [Path("/content"), Path("/content/drive/MyDrive"), Path("/content/drive/MyDrive/nids_data")]:
        if base.exists():
            zip_candidates.extend(sorted(base.glob("*.zip")))
    print("Zip candidates:", [str(path) for path in zip_candidates])
    for zip_path in zip_candidates:
        if zip_contains_expected(zip_path):
            print("Extracting CICIDS2017 raw CSV files from:", zip_path)
            extract_expected_files(zip_path, Path("/content/cicids2017_raw"))
            CICIDS_RAW_DIR = Path("/content/cicids2017_raw")
            break

print("Selected raw dir after zip check:", CICIDS_RAW_DIR)

Selected raw dir after zip check: /content/drive/MyDrive/cicids2017


In [13]:
from dataclasses import dataclass

def find_present_files(raw_dir: Path):
    """Returns a dict of {filename: Path} for all files in raw_dir."""
    return {p.name: p for p in raw_dir.glob("*") if p.is_file()}

@dataclass
class FileProbeReport:
    path: Path
    exists: bool
    ok: bool
    stat_size: int = 0
    readable_size: int = 0
    line_count: int = 0
    message: str = ""


def probe_file(path: Path, chunk_size: int = 1024 * 1024):
    path = Path(path)
    if not path.exists():
        return FileProbeReport(path=path, exists=False, ok=False, message="missing")
    try:
        stat_size = path.stat().st_size
        readable_size = 0
        line_count = 0
        with path.open("rb") as handle:
            for chunk in iter(lambda: handle.read(chunk_size), b""):
                readable_size += len(chunk)
                line_count += chunk.count(b"\n")
    except OSError as exc:
        return FileProbeReport(path=path, exists=True, ok=False, message=f"read error: {exc}")
    ok = readable_size == stat_size
    message = "ok" if ok else "readable bytes differ from filesystem stat size"
    return FileProbeReport(
        path=path,
        exists=True,
        ok=ok,
        stat_size=stat_size,
        readable_size=readable_size,
        line_count=line_count,
        message=message,
    )


if CICIDS_RAW_DIR is None:
    print("No complete CICIDS2017 raw directory found.")
    print("Put the eight CSV files in one of these folders, then rerun configure/extract/verify cells:")
    for candidate in CANDIDATE_RAW_DIRS:
        print("-", candidate)
    print("Missing file names:")
    for name in EXPECTED_FILES:
        print("-", name)
    raise FileNotFoundError("CICIDS2017 raw CSV files are not available in Colab yet.")

PRESENT_RAW_FILES = find_present_files(CICIDS_RAW_DIR)
missing = [name for name in EXPECTED_FILES if name not in PRESENT_RAW_FILES]
print(f"Checking directory: {CICIDS_RAW_DIR}")
if missing:
    print("Missing files:")
    for name in missing:
        print("-", name)
    raise FileNotFoundError(f"Missing {len(missing)} files in {CICIDS_RAW_DIR}")

failed = []
for expected in EXPECTED_FILES:
    path = PRESENT_RAW_FILES[expected]
    report = probe_file(path)
    print(
        f"{expected}: actual={path.name} ok={report.ok} "
        f"stat_MB={report.stat_size / 1024 / 1024:.3f} "
        f"read_MB={report.readable_size / 1024 / 1024:.3f} "
        f"lines={report.line_count}"
    )
    if not report.ok:
        failed.append(path.name)

if failed:
    raise RuntimeError("Unreadable or incomplete raw CSV files: " + ", ".join(failed))

print("All raw files verified.")

Checking directory: /content/drive/MyDrive/cicids2017
Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv: actual=Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv ok=True stat_MB=73.551 read_MB=73.551 lines=225746
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv: actual=Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv ok=True stat_MB=73.343 read_MB=73.343 lines=286468
Friday-WorkingHours-Morning.pcap_ISCX.csv: actual=Friday-WorkingHours-Morning.pcap_ISCX.csv ok=True stat_MB=55.615 read_MB=55.615 lines=191034
Monday-WorkingHours.pcap_ISCX.csv: actual=Monday-WorkingHours.pcap_ISCX.csv ok=True stat_MB=168.732 read_MB=168.732 lines=529919
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv: actual=Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv ok=True stat_MB=79.253 read_MB=79.253 lines=288603
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv: actual=Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv ok=True stat_MB=49.613 read_MB=49.613 lines=170367
Tues

In [14]:
import yaml

config_path = PROJECT_DIR / "config" / "datasets.yaml"
config = yaml.safe_load(config_path.read_text(encoding="utf-8"))
config["datasets"]["cicids2017"]["raw_dir"] = str(CICIDS_RAW_DIR)
config["datasets"]["cicids2017"]["cache_path"] = str(CICIDS_CACHE_PATH)
config_path.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")

print(config["datasets"]["cicids2017"])

{'name': 'CICIDS2017', 'raw_dir': '/content/drive/MyDrive/cicids2017', 'cache_path': '/content/cicids2017_cache/merged.csv', 'remote_url': None, 'label_column': 'label', 'categorical_columns': []}


In [15]:
import gc
import json
import time

import joblib
import yaml

from src.models.ml_baselines import (
    build_lightgbm,
    build_logistic_regression,
    build_random_forest,
    build_xgboost,
)
from src.pipeline.baseline_runner import build_adapter, save_artifacts
from src.utils.seed import set_seed

root = PROJECT_DIR
datasets_config = yaml.safe_load((root / "config" / "datasets.yaml").read_text(encoding="utf-8"))
experiments_config = yaml.safe_load((root / "config" / "experiments.yaml").read_text(encoding="utf-8"))
seed = int(experiments_config.get("seed", 42))
set_seed(seed)

split_cfg = experiments_config.get("split", {})
adapter = build_adapter("cicids2017", datasets_config, root, seed)
output = adapter.preprocess(
    classification_type=CLASSIFICATION,
    test_size=float(split_cfg.get("test_size", 0.2)),
    val_size=float(split_cfg.get("val_size", 0.2)),
    scaler_type=experiments_config.get("preprocessing", {}).get("scaler", "standard"),
)

artifact_dir = root / "artifacts" / "cicids2017" / CLASSIFICATION
save_artifacts(output, artifact_dir)

builders = {
    "LogisticRegression": lambda: build_logistic_regression(seed),
    "RandomForest": lambda: build_random_forest(seed),
    "XGBoost": lambda: build_xgboost(seed),
    "LightGBM": lambda: build_lightgbm(seed),
}

manifest = {
    "dataset": "cicids2017",
    "classification": CLASSIFICATION,
    "mode": "train_eval",
    "seed": seed,
    "feature_count": len(output.feature_names),
    "label_mapping": output.label_mapping,
    "split_shapes": {
        "train": list(output.X_train.shape),
        "val": list(output.X_val.shape),
        "test": list(output.X_test.shape),
    },
    "models": {},
}

for name in MODELS_TO_TRAIN:
    if name not in builders:
        raise ValueError(f"Unknown model: {name}")
    model = builders[name]()
    start = time.time()
    print(f"Training {name}...")
    model.fit(output.X_train, output.y_train)
    model_path = artifact_dir / f"{name}.pkl"
    joblib.dump(model, model_path)
    elapsed = time.time() - start
    manifest["models"][name] = {
        "artifact": str(model_path.relative_to(root)),
        "train_seconds": elapsed,
    }
    print(f"Saved {name} to {model_path} in {elapsed:.1f}s")
    del model
    gc.collect()

results_dir = root / "results"
results_dir.mkdir(parents=True, exist_ok=True)
manifest_path = results_dir / "cicids2017_multi_ml_train_eval_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(f"Saved train/eval manifest to {manifest_path}")

Training LogisticRegression...
Saved LogisticRegression to /content/Nids_deep_learning/ids_deep_learning/artifacts/cicids2017/multi/LogisticRegression.pkl in 749.3s
Training RandomForest...
Saved RandomForest to /content/Nids_deep_learning/ids_deep_learning/artifacts/cicids2017/multi/RandomForest.pkl in 1240.9s
Training XGBoost...
Saved XGBoost to /content/Nids_deep_learning/ids_deep_learning/artifacts/cicids2017/multi/XGBoost.pkl in 706.8s
Training LightGBM...
Saved LightGBM to /content/Nids_deep_learning/ids_deep_learning/artifacts/cicids2017/multi/LightGBM.pkl in 532.1s
Saved train/eval manifest to /content/Nids_deep_learning/ids_deep_learning/results/cicids2017_multi_ml_train_eval_manifest.json


## Evaluate ML Baselines

This cell evaluates all trained CICIDS2017 ML artifacts on the held-out test split from the preprocessing cell. It writes `results/cicids2017_multi_ml_baselines.json`, including `train_seconds` and `eval_seconds` for each model.

In [16]:
import pandas as pd

from src.evaluation import compute_classification_metrics
from src.pipeline.baseline_runner import benign_label_index

ml_eval_report = {
    "dataset": "cicids2017",
    "classification_type": CLASSIFICATION,
    "evaluation_mode": "colab_train_eval",
    "seed": seed,
    "n_features": len(output.feature_names),
    "label_mapping": output.label_mapping,
    "split_shapes": {
        "train": list(output.X_train.shape),
        "val": list(output.X_val.shape),
        "test": list(output.X_test.shape),
    },
    "models": {},
}

benign = benign_label_index(output.label_mapping)
summary_rows = []

for name in MODELS_TO_TRAIN:
    model_path = artifact_dir / f"{name}.pkl"
    if not model_path.exists():
        raise FileNotFoundError(f"Missing trained artifact: {model_path}")

    print(f"Evaluating {name}...")
    model = joblib.load(model_path)
    start = time.time()
    pred = model.predict(output.X_test)
    probs = model.predict_proba(output.X_test) if hasattr(model, "predict_proba") else None
    eval_seconds = time.time() - start

    metrics = compute_classification_metrics(
        output.y_test,
        pred,
        y_probs=probs,
        benign_label=benign,
    )
    metrics["train_seconds"] = manifest["models"].get(name, {}).get("train_seconds")
    metrics["eval_seconds"] = eval_seconds
    metrics["artifact"] = str(model_path.relative_to(root))
    ml_eval_report["models"][name] = metrics

    manifest["models"].setdefault(name, {})
    manifest["models"][name].update({
        "eval_seconds": eval_seconds,
        "accuracy": metrics.get("accuracy"),
        "macro_f1": metrics.get("macro_f1"),
        "weighted_f1": metrics.get("weighted_f1"),
        "far": metrics.get("far"),
    })

    summary_rows.append({
        "model": name,
        "accuracy": metrics.get("accuracy"),
        "macro_f1": metrics.get("macro_f1"),
        "weighted_f1": metrics.get("weighted_f1"),
        "far": metrics.get("far"),
        "train_seconds": metrics.get("train_seconds"),
        "eval_seconds": eval_seconds,
    })
    print(
        f"{name}: macro_f1={metrics['macro_f1']:.6f}, "
        f"weighted_f1={metrics['weighted_f1']:.6f}, "
        f"far={metrics['far']:.6f}, eval_seconds={eval_seconds:.1f}"
    )
    del model
    gc.collect()

ml_eval_path = results_dir / "cicids2017_multi_ml_baselines.json"
ml_eval_path.write_text(json.dumps(ml_eval_report, indent=2), encoding="utf-8")
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

summary_df = pd.DataFrame(summary_rows).sort_values(["macro_f1", "far"], ascending=[False, True])
print(f"Saved ML evaluation report to {ml_eval_path}")
print(f"Updated train/eval manifest at {manifest_path}")
display(summary_df)

Evaluating LogisticRegression...
LogisticRegression: macro_f1=0.556001, weighted_f1=0.904778, far=0.169114, eval_seconds=0.8
Evaluating RandomForest...
RandomForest: macro_f1=0.976169, weighted_f1=0.998542, far=0.000699, eval_seconds=24.0
Evaluating XGBoost...
XGBoost: macro_f1=0.892134, weighted_f1=0.998873, far=0.000925, eval_seconds=33.5
Evaluating LightGBM...


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


LightGBM: macro_f1=0.279609, weighted_f1=0.497171, far=0.673153, eval_seconds=60.5
Saved ML evaluation report to /content/Nids_deep_learning/ids_deep_learning/results/cicids2017_multi_ml_baselines.json
Updated train/eval manifest at /content/Nids_deep_learning/ids_deep_learning/results/cicids2017_multi_ml_train_eval_manifest.json


,model,accuracy,macro_f1,weighted_f1,far,train_seconds,eval_seconds
1,RandomForest,0.998551,0.976169,0.998542,0.000699,1240.924326,23.998933
2,XGBoost,0.998876,0.892134,0.998873,0.000925,706.807218,33.525561
0,LogisticRegression,0.858214,0.556001,0.904778,0.169114,749.279134,0.782403
3,LightGBM,0.373080,0.279609,0.497171,0.673153,532.068476,60.486040


In [17]:
import joblib

artifact_files = [
    artifact_dir / "scaler.pkl",
    artifact_dir / "encoders.pkl",
    artifact_dir / "imputer.pkl",
    artifact_dir / "label_mapping.json",
    artifact_dir / "inference_config.json",
] + [artifact_dir / f"{name}.pkl" for name in MODELS_TO_TRAIN]

for path in artifact_files:
    report = probe_file(path)
    print(path.relative_to(PROJECT_DIR), report.ok, report.readable_size)
    if not report.ok:
        raise RuntimeError(f"Artifact is incomplete: {path}")
    if path.suffix == ".pkl":
        joblib.load(path)

print("Artifact verification complete.")

artifacts/cicids2017/multi/scaler.pkl True 3847
artifacts/cicids2017/multi/encoders.pkl True 597
artifacts/cicids2017/multi/imputer.pkl True 2599
artifacts/cicids2017/multi/label_mapping.json True 152
artifacts/cicids2017/multi/inference_config.json True 1802
artifacts/cicids2017/multi/LogisticRegression.pkl True 6047
artifacts/cicids2017/multi/RandomForest.pkl True 160292289
artifacts/cicids2017/multi/XGBoost.pkl True 2923866
artifacts/cicids2017/multi/LightGBM.pkl True 4355948
Artifact verification complete.


In [18]:
import zipfile

OUTPUT_ZIP.parent.mkdir(parents=True, exist_ok=True)
if OUTPUT_ZIP.exists():
    OUTPUT_ZIP.unlink()

extra_result_files = [manifest_path]
if "ml_eval_path" in globals() and ml_eval_path.exists():
    extra_result_files.append(ml_eval_path)
else:
    fallback_eval_path = PROJECT_DIR / "results" / "cicids2017_multi_ml_baselines.json"
    if fallback_eval_path.exists():
        extra_result_files.append(fallback_eval_path)
    else:
        print("Warning: ML evaluation JSON not found. Run the evaluation cell before zipping.")

with zipfile.ZipFile(OUTPUT_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(artifact_dir.glob("*")):
        if path.is_file():
            archive.write(path, path.relative_to(PROJECT_DIR))
    for path in extra_result_files:
        if path.exists():
            archive.write(path, path.relative_to(PROJECT_DIR))

print(f"Wrote {OUTPUT_ZIP}")
print("Included result files:")
for path in extra_result_files:
    print("-", path.relative_to(PROJECT_DIR), path.exists())

Wrote /content/drive/MyDrive/nids_outputs/cicids2017_ml_artifacts.zip
Included result files:
- results/cicids2017_multi_ml_train_eval_manifest.json True
- results/cicids2017_multi_ml_baselines.json True
